# A Clinician's Audit of GiBleed

### What a physician sees in the OMOP dataset that everyone learns on

**Andrew O. Cole**, MD, MSc, PhD (Biomathematics) · *self-directed, 2026*

---

**GiBleed** is the default teaching database for [Eunomia](https://ohdsi.github.io/Eunomia/) — the sandbox where most people first learn the OHDSI (pronounced *"Odyssey"*) / OMOP Common Data Model. It is small, fast, portable, and conformant, which is exactly why it became the place nearly every newcomer's mental model of *"what OMOP data looks like"* is formed.

This notebook audits it using nothing but SQL and clinical reasoning. It runs top‑to‑bottom in about ninety seconds on a fresh Eunomia install.

## What it finds

GiBleed passes automated data-quality checks. It also:

- gives **100% of its 2,694 people osteoarthritis** — exactly one record each, ages 31–47, no exceptions
- hides an **8‑condition "GI‑bleed module"**: seven conditions are the textbook differential diagnosis of GI bleeding, each occurring **exactly once per person**, plus osteoarthritis as the indication
- records **zero deaths** in a population whose oldest recorded age is **110**
- makes **68%** of everyone a celecoxib user (real world: ~1–3%), one exposure record each
- orders drug exposure before GI bleed **355 times out of 355** — not one reversal, not one same‑day tie

**None of these are bugs.** They are scaffolding: GiBleed was built to teach one thing — the NSAID → GI‑bleed method — and every item above is holding that lesson up. For its intended purpose the dataset is excellent.

**None of them violate a range check.** Every value is well-formed, in-range, type-correct, and conformant. These findings are only implausible if you know what osteoarthritis does to a knee over forty years.

**That's the finding.**

## How to read this notebook

- **§1 Environment** installs and loads the stack.
- **§2 Orient** counts the population and — before trusting any arithmetic — checks how dates are stored.
- **§3 The audit** is the substance: the one‑shot module, the osteoarthritis scaffold, the missing mortality compartment, the drug landscape, the outcome and its drawn arrow, and the metadata that confirms the reconstruction.

The single most important query is **the one‑shot module table in §3.1**. Everything after it is elaboration.

> The mathematics tells me what shape a disease should leave in data.
> The medicine tells me when the shape is wrong.


## 1 · Environment

In [ ]:
# Notebook version (for reproducibility)
R.version.string
basename(.libPaths())

*Windows only:* building packages from source needs Rtools. If `pkgbuild::has_build_tools()` returns `FALSE`, install Rtools first. On macOS/Linux this is not required — the install cell below falls back to source automatically.

In [ ]:
# Install only what this notebook uses, with retries for flaky mirrors
required <- c("Eunomia", "DBI", "RSQLite")
missing  <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]

if (length(missing)) {
    options(repos = c(CRAN = "https://cloud.r-project.org"),
            download.file.method = "libcurl", timeout = 300)
    for (i in 1:5) {
        install.packages(missing, dependencies = TRUE)
        missing <- missing[!vapply(missing, requireNamespace, logical(1), quietly = TRUE)]
        if (!length(missing)) break
        message("retry ", i, " - still missing: ", paste(missing, collapse = ", "))
        Sys.sleep(3)
    }
}
stopifnot("packages failed to install" = !length(missing))

In [ ]:
# Download + extract GiBleed, then open one connection we reuse throughout
invisible(capture.output(suppressMessages(suppressWarnings({
  library(Eunomia); library(DBI); library(RSQLite)
  connectionDetails <- getEunomiaConnectionDetails()
}))))
con <- dbConnect(RSQLite::SQLite(), connectionDetails$server())
cat("Eunomia ready - GiBleed downloaded and connection open.\n")

## 2 · Orient in the database (and check before trusting it)

In [ ]:
# How many people? (the denominator for everything below)
dbGetQuery(con, "SELECT COUNT(*) AS n_persons FROM person;")

In [ ]:
# What tables exist, how big is the condition table, and what are the commonest conditions?
dbListTables(con)

dbGetQuery(con, "
  SELECT COUNT(*) AS n_conditions,
         COUNT(DISTINCT person_id) AS n_people
  FROM condition_occurrence;
")

dbGetQuery(con, "
  SELECT c.concept_name, COUNT(*) AS n
  FROM condition_occurrence co
  JOIN concept c ON co.condition_concept_id = c.concept_id
  GROUP BY c.concept_name
  ORDER BY n DESC
  LIMIT 10;
")

In [ ]:
# CHECK THE COLUMN BEFORE TRUSTING ARITHMETIC ON IT.
# Eunomia's SQLite stores dates as Unix-epoch INTEGERS, not the CDM-specified DATE type.
# That is why every age calculation below uses strftime('%Y', col, 'unixepoch').
dbGetQuery(con, "SELECT condition_start_date FROM condition_occurrence LIMIT 5;")  # integers, not 'YYYY-MM-DD'
dbGetQuery(con, "SELECT year_of_birth FROM person LIMIT 5;")

## 3 · The audit

### 3.1 The one-shot module — the single most important query

Read `recs_per_person` as a measurement of a disease's *state graph*: how many times the average affected person is recorded in that state. A chronic or recurrent disease leaves **many** footprints (viral sinusitis: 6.43). A value of **exactly 1.00** is a state entered once and never re-entered — the wrong topological shape for anything chronic, recurrent, or progressive.

The first query ranks every common condition by `recs_per_person`; the eight at the `1.00` floor are easy to miss at the bottom of a long list, so the **second query isolates them** and labels each by its role. Read as a clinician, seven of the eight are the differential diagnosis of GI bleeding.

In [ ]:
# Every condition with >=300 people, ranked by records-per-person.
# The 'life-like' conditions (recurrence) are at the top; the eight welded to the 1.00 floor are at the bottom.
dbGetQuery(con, "
SELECT c.concept_name,
       COUNT(*) AS n_records,
       COUNT(DISTINCT co.person_id) AS n_people,
       ROUND(COUNT(*)*1.0 / COUNT(DISTINCT co.person_id), 2) AS recs_per_person,
       MIN(CAST(strftime('%Y', co.condition_start_date, 'unixepoch') AS INTEGER) - p.year_of_birth) AS min_age,
       MAX(CAST(strftime('%Y', co.condition_start_date, 'unixepoch') AS INTEGER) - p.year_of_birth) AS max_age
FROM condition_occurrence co
JOIN concept c ON co.condition_concept_id = c.concept_id
JOIN person  p ON co.person_id = p.person_id
GROUP BY c.concept_name
HAVING COUNT(DISTINCT co.person_id) >= 300
ORDER BY recs_per_person DESC;
")

In [ ]:
# ISOLATE THE MODULE: the eight conditions at recs_per_person = 1.00, labeled by clinical role.
# This reproduces the audit's centerpiece table directly from the data.
dbGetQuery(con, "
SELECT
  CASE c.concept_name
    WHEN 'Peptic ulcer'               THEN 'Upper GI (cause)'
    WHEN 'Esophagitis'                THEN 'Upper GI (cause)'
    WHEN 'Angiodysplasia of stomach'  THEN 'Upper GI (cause)'
    WHEN 'Diverticular disease'       THEN 'Lower GI (cause)'
    WHEN 'Polyp of colon'             THEN 'Lower GI (cause)'
    WHEN 'Ulcerative colitis'         THEN 'Lower GI (cause)'
    WHEN 'Gastrointestinal hemorrhage' THEN 'OUTCOME'
    WHEN 'Osteoarthritis'             THEN 'INDICATION'
  END AS module_role,
  c.concept_name,
  COUNT(DISTINCT co.person_id) AS n_people,
  ROUND(COUNT(DISTINCT co.person_id)*100.0 / 2694, 1) AS prevalence_pct,
  ROUND(COUNT(*)*1.0 / COUNT(DISTINCT co.person_id), 2) AS recs_per_person,
  MIN(CAST(strftime('%Y', co.condition_start_date, 'unixepoch') AS INTEGER) - p.year_of_birth) AS min_age,
  MAX(CAST(strftime('%Y', co.condition_start_date, 'unixepoch') AS INTEGER) - p.year_of_birth) AS max_age
FROM condition_occurrence co
JOIN concept c ON co.condition_concept_id = c.concept_id
JOIN person  p ON co.person_id = p.person_id
GROUP BY c.concept_name
HAVING COUNT(DISTINCT co.person_id) >= 300
   AND ROUND(COUNT(*)*1.0 / COUNT(DISTINCT co.person_id), 2) = 1.00
ORDER BY (c.concept_name = 'Osteoarthritis') DESC,
         (c.concept_name = 'Gastrointestinal hemorrhage') DESC,
         n_people DESC;
")

Two independent rulers — `recs_per_person` and the age envelope — were measured separately and select the **same eight conditions**. Every module member is diagnosed inside a ~15‑year midlife window and never after **47**; peptic ulcer runs a distinct, younger gate (24–34). The positive control below proves this is a true absence, not censoring.

In [ ]:
# POSITIVE CONTROL: viral sinusitis is recorded across the whole lifespan (0-109).
# So the recorder works to age 109 -> the module's silence after 47 is a real absence, not missing follow-up.
dbGetQuery(con, "
SELECT MIN(CAST(strftime('%Y', co.condition_start_date, 'unixepoch') AS INTEGER) - p.year_of_birth) AS min_age,
       MAX(CAST(strftime('%Y', co.condition_start_date, 'unixepoch') AS INTEGER) - p.year_of_birth) AS max_age
FROM condition_occurrence co
JOIN concept c ON co.condition_concept_id = c.concept_id
JOIN person  p ON co.person_id = p.person_id
WHERE c.concept_name = 'Viral sinusitis';
")

### 3.2 Osteoarthritis is the eligibility scaffold, not a modelled disease

100% prevalence is not a data point; it is a claim about a population — *everyone here is old enough to have worn out a joint.* With zero variance, osteoarthritis cannot confound, mediate, stratify, or be adjusted for. Every comorbidity overlap involving it is therefore **arithmetically forced**: it is the sample space, so every other condition is a subset of it.

In [ ]:
# Osteoarthritis: exactly one record per person, 100% of the population, ages 31-47.
dbGetQuery(con, "
SELECT COUNT(*) AS n_records,
       COUNT(DISTINCT co.person_id) AS n_people,
       ROUND(COUNT(DISTINCT co.person_id)*100.0 / 2694, 1) AS prevalence_pct,
       MIN(CAST(strftime('%Y', co.condition_start_date, 'unixepoch') AS INTEGER) - p.year_of_birth) AS min_age,
       MAX(CAST(strftime('%Y', co.condition_start_date, 'unixepoch') AS INTEGER) - p.year_of_birth) AS max_age
FROM condition_occurrence co
JOIN concept c ON co.condition_concept_id = c.concept_id
JOIN person  p ON co.person_id = p.person_id
WHERE c.concept_name = 'Osteoarthritis';
")

In [ ]:
# The forced overlap: everyone with otitis media also 'has' osteoarthritis --
# not a comorbidity discovery, an identity. 2,025 = ALL 2,025 otitis patients.
dbGetQuery(con, "
SELECT COUNT(DISTINCT a.person_id) AS people_with_both
FROM condition_occurrence a
JOIN concept ca ON a.condition_concept_id = ca.concept_id
JOIN condition_occurrence b ON a.person_id = b.person_id
JOIN concept cb ON b.condition_concept_id = cb.concept_id
WHERE ca.concept_name = 'Otitis media'
  AND cb.concept_name = 'Osteoarthritis';
")

### 3.3 No mortality compartment

Death is the canonical absorbing state of any human‑population model. Here it is missing entirely: the `death` table is empty, yet people are recorded up to age **110**. (`n_over_100` uses birth year against a 2020 reference, so it is an approximate head‑count; `max_recorded_age` is the oldest age at which any condition is actually recorded — the meaningful ceiling, and there is no death behind it.)

In [ ]:
dbGetQuery(con, "
SELECT
  (SELECT COUNT(*) FROM death) AS n_deaths,
  (SELECT COUNT(*) FROM person WHERE (2020 - year_of_birth) > 100) AS n_over_100_by_birthyear,
  (SELECT MAX(CAST(strftime('%Y', co.condition_start_date, 'unixepoch') AS INTEGER) - p.year_of_birth)
     FROM condition_occurrence co JOIN person p ON co.person_id = p.person_id) AS max_recorded_age;
")

### 3.4 The drug landscape does not resemble prescribing

Celecoxib is a second‑line, cardiovascular‑risk‑carrying COX‑2 inhibitor; in the real world it is 1–3% of a population and channelled to the already‑high‑risk. Here **68%** are on it, one record each — chronic therapy compressed to a single event, and the channelling that drives real‑world confounding by indication simply absent.

In [ ]:
# Commonest drugs (celecoxib is not even top-5 -- vaccines and acetaminophen dominate) ...
dbGetQuery(con, "
SELECT c.concept_name, COUNT(DISTINCT de.person_id) AS n_people
FROM drug_exposure de
JOIN concept c ON de.drug_concept_id = c.concept_id
GROUP BY c.concept_name
ORDER BY n_people DESC
LIMIT 10;
")

# ... yet celecoxib exposure is population-scale: 1,844 users = 68% of 2,694, one exposure record each.
dbGetQuery(con, "
SELECT COUNT(*) AS n_records,
       COUNT(DISTINCT de.person_id) AS n_users,
       ROUND(COUNT(DISTINCT de.person_id)*100.0 / 2694, 1) AS pct_of_population
FROM drug_exposure de
JOIN concept c ON de.drug_concept_id = c.concept_id
WHERE c.concept_name = 'celecoxib';
")

### 3.5 The outcome, and an arrow that was drawn rather than observed

Gastrointestinal hemorrhage (concept 192671) is the outcome: 479 people, one record each. Of the 479, **355 are on celecoxib**, and the drug precedes the bleed **355 / 355**. The `>=` in the ordering test folds any same‑day pair into `bleed_first`, so `bleed_first = 0` proves **zero reversals AND zero same‑day ties** — a perfection real, noisy data never shows.

In [ ]:
# The outcome: one record per person.
dbGetQuery(con, "
SELECT COUNT(*) AS n_records, COUNT(DISTINCT person_id) AS n_people
FROM condition_occurrence
WHERE condition_concept_id = 192671;
")

# Bleeders on celecoxib, and strict temporal ordering (drug before bleed).
dbGetQuery(con, "
SELECT COUNT(DISTINCT co.person_id) AS bleeders_on_celecoxib
FROM condition_occurrence co
JOIN drug_exposure de ON co.person_id = de.person_id
JOIN concept c ON de.drug_concept_id = c.concept_id
WHERE co.condition_concept_id = 192671 AND c.concept_name = 'celecoxib';
")

dbGetQuery(con, "
SELECT SUM(CASE WHEN de.drug_exposure_start_date <  co.condition_start_date THEN 1 ELSE 0 END) AS drug_first,
       SUM(CASE WHEN de.drug_exposure_start_date >= co.condition_start_date THEN 1 ELSE 0 END) AS bleed_first
FROM condition_occurrence co
JOIN drug_exposure de ON co.person_id = de.person_id
JOIN concept c ON de.drug_concept_id = c.concept_id
WHERE co.condition_concept_id = 192671 AND c.concept_name = 'celecoxib';
")

The 2×2 below gives RR = 1.32, OR = 1.40. The arithmetic is correct — and it measures nothing about celecoxib. With no confounding generated (no channelling) and no noise (ordering is 355/355), the association is entirely the generator's parameter setting, not biology. This is why *a method "validated against GiBleed" has not been validated*: there is no bias to remove, so a correct and a broken method return the same number.

In [ ]:
# The full 2x2 (celecoxib x GI-bleed) as a labeled contingency table, with RR and OR.
cells_2x2 <- dbGetQuery(con, "
WITH bleeders AS (SELECT DISTINCT person_id FROM condition_occurrence WHERE condition_concept_id = 192671),
     cele     AS (SELECT DISTINCT de.person_id FROM drug_exposure de
                  JOIN concept c ON de.drug_concept_id = c.concept_id
                  WHERE c.concept_name = 'celecoxib')
SELECT
  SUM(CASE WHEN e.person_id IS NOT NULL AND b.person_id IS NOT NULL THEN 1 ELSE 0 END) AS exposed_bled,
  SUM(CASE WHEN e.person_id IS NOT NULL AND b.person_id IS NULL     THEN 1 ELSE 0 END) AS exposed_no_bleed,
  SUM(CASE WHEN e.person_id IS NULL     AND b.person_id IS NOT NULL THEN 1 ELSE 0 END) AS unexposed_bled,
  SUM(CASE WHEN e.person_id IS NULL     AND b.person_id IS NULL     THEN 1 ELSE 0 END) AS unexposed_no_bleed
FROM person p
LEFT JOIN cele     e ON p.person_id = e.person_id
LEFT JOIN bleeders b ON p.person_id = b.person_id;
")

# pull the four cell counts (avoid naming a variable 'c' -- it shadows base::c())
eb  <- cells_2x2$exposed_bled;      enb <- cells_2x2$exposed_no_bleed
ub  <- cells_2x2$unexposed_bled;    unb <- cells_2x2$unexposed_no_bleed

# tidy 2x2 with row/column margins: rows = celecoxib exposure, cols = GI-bleed outcome
tab_2x2 <- data.frame(
  "GI bleed"    = c(eb,        ub,        eb + ub),
  "No GI bleed" = c(enb,       unb,       enb + unb),
  "Total"       = c(eb + enb,  ub + unb,  eb + enb + ub + unb),
  row.names     = c("Celecoxib exposed", "Celecoxib unexposed", "Total"),
  check.names   = FALSE
)

RR <- (eb / (eb + enb)) / (ub / (ub + unb))
OR <- (eb * unb) / (enb * ub)

cat(sprintf("Naive 2x2:  RR = %.2f   OR = %.2f\n", RR, OR))
cat("(No confounding generated + no noise -> this reflects the generator's\n parameter setting, not biology, not bias, not celecoxib.)\n\n")

tab_2x2   # last expression -> renders as a clean HTML table

### 3.6 The metadata confirms the reconstruction

Everything above was recovered from output alone. The `cdm_source` table — exemplary provenance metadata — independently confirms the generator and version. Note what it does *not* contain: any field for **fitness for purpose**. It answers "where did this come from?" but has no field for "what can I use it for?" 

In [ ]:
# Provenance: Synthea, CDM v5.3.1, released 2019-05-25, vocabulary v5.0 (18-JAN-19).
dbGetQuery(con, "
SELECT cdm_source_abbreviation, cdm_holder, cdm_version, vocabulary_version,
       source_release_date, cdm_etl_reference
FROM cdm_source;
")

## 4 · Close

In [ ]:
dbDisconnect(con)
sessionInfo()